# 07 — Resolution-Matched Control Experiment

**Milestone 6** (August 2026)

Every generalisation evaluation pairs high-resolution generator fakes with CIFAKE
`test/REAL` images (32×32 upscaled to 224). Resolution is therefore confounded with
the class label. This notebook runs a 2×2 control to test whether the detector is
separating **resampling signatures** rather than **generator artefacts**.

| Condition | Images | Pipeline | Question |
|-----------|--------|----------|----------|
| **A** | CIFAKE REAL | Native (`Resize(256)` + `CenterCrop(224)`) | Reference real FAKE-rate |
| **B** | High-res real photos (COCO / matched-source) | Native | Do high-res reals trigger false positives? |
| **C** | Same high-res reals | Force 32×32, then native | Isolates resolution with content held fixed |
| **D** | Generator FAKEs (5 families) | Force 32×32, then native | Does fake detection survive resolution matching? |

**Interpretation:**
- High FAKE-rate in **B** → resolution confound; the 94–97% generalisation figures are inflated.
- Low FAKE-rate in **B** (comparable to **A**) → detector genuinely transfers; confound ruled out.


## 0. Colab Setup

In [ ]:
# Cell 1 — Mount Google Drive & define paths
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    DRIVE_ROOT = "/content/drive/MyDrive/ai-image-detection"
    REPO_DIR = "/content/ai-image-detection"
    DATA_DIR = os.path.join(REPO_DIR, "data", "raw", "cifake")
    CHECKPOINT_DIR = os.path.join(DRIVE_ROOT, "checkpoints")
    OUTPUT_DIR = os.path.join(DRIVE_ROOT, "outputs")
    GEN_DIR = os.path.join(DRIVE_ROOT, "data", "generalisation")
    CONTROLS_DIR = os.path.join(GEN_DIR, "_controls")

    os.makedirs(DRIVE_ROOT, exist_ok=True)
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    if not os.path.exists(REPO_DIR):
        !git clone https://github.com/krishi-shah/ai-image-detection.git {REPO_DIR}
    else:
        print(f"Repo already present at {REPO_DIR}")
else:
    REPO_DIR = str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
    DRIVE_ROOT = REPO_DIR
    DATA_DIR = os.path.join(REPO_DIR, "data", "raw", "cifake")
    CHECKPOINT_DIR = os.path.join(REPO_DIR, "outputs", "checkpoints")
    OUTPUT_DIR = os.path.join(REPO_DIR, "outputs")
    GEN_DIR = os.path.join(REPO_DIR, "data", "generalisation")
    CONTROLS_DIR = os.path.join(GEN_DIR, "_controls")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

print(f"Working directory: {os.getcwd()}")
print(f"Data directory:    {DATA_DIR}")
print(f"Checkpoint dir:    {CHECKPOINT_DIR}")
print(f"Controls dir:      {CONTROLS_DIR}")
print(f"Output dir:        {OUTPUT_DIR}")


In [ ]:
# Cell 2 — Install dependencies
!pip install -q -r requirements.txt


In [ ]:
# Cell 3 — Extract CIFAKE from Drive zip (Colab only)
import os, zipfile, glob

os.makedirs(DATA_DIR, exist_ok=True)

if os.path.exists(os.path.join(DATA_DIR, "train")):
    print("CIFAKE already extracted — skipping.")
else:
    zip_candidates = glob.glob(os.path.join(DRIVE_ROOT, "*.zip"))
    if not zip_candidates:
        raise FileNotFoundError(
            f"No .zip found in {DRIVE_ROOT}. Upload the CIFAKE archive there first."
        )
    zip_path = zip_candidates[0]
    print(f"Found zip: {zip_path}")
    print("Extracting (this takes ~1-2 minutes)...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(DATA_DIR)
    # Flatten if archive nested an extra folder
    for split in ["train", "test"]:
        nested = glob.glob(os.path.join(DATA_DIR, "*", split))
        if nested and not os.path.exists(os.path.join(DATA_DIR, split)):
            import shutil
            parent = os.path.dirname(nested[0])
            for item in os.listdir(parent):
                src = os.path.join(parent, item)
                dst = os.path.join(DATA_DIR, item)
                if not os.path.exists(dst):
                    shutil.move(src, dst)
    print("Extraction complete.")

print("Contents of DATA_DIR:", os.listdir(DATA_DIR))


## 1. Load Model & Baseline Temperature

In [ ]:
# Cell 4 — Load checkpoint and temperature
import json
import torch
from pathlib import Path

from src.model.detector import build_detector, load_checkpoint

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

ckpt_candidates = [
    os.path.join(CHECKPOINT_DIR, "best_detector.pth"),
    os.path.join(OUTPUT_DIR, "checkpoints", "best_detector.pth"),
    os.path.join(REPO_DIR, "outputs", "checkpoints", "best_detector.pth"),
]
ckpt_path = next((p for p in ckpt_candidates if os.path.exists(p)), None)
if ckpt_path is None:
    raise FileNotFoundError(f"No checkpoint found. Tried: {ckpt_candidates}")

model = build_detector(pretrained=False)
info = load_checkpoint(model, ckpt_path)
model = model.to(device)
model.eval()
print(f"Model loaded from: {ckpt_path} (epoch {info.get('epoch', '?')})")

baseline_candidates = [
    os.path.join(OUTPUT_DIR, "results", "baseline_results.json"),
    os.path.join(DRIVE_ROOT, "outputs", "results", "baseline_results.json"),
    os.path.join(REPO_DIR, "outputs", "results", "baseline_results.json"),
]
baseline_path = next((p for p in baseline_candidates if os.path.exists(p)), None)
TEMPERATURE = 1.2189
if baseline_path:
    with open(baseline_path) as f:
        baseline = json.load(f)
    TEMPERATURE = baseline.get("temperature", TEMPERATURE)
    print(f"Baseline accuracy: {baseline.get('test_accuracy')}")
    print(f"Temperature: {TEMPERATURE}")
else:
    print(f"Baseline JSON not found; using T={TEMPERATURE}")

CIFAKE_REAL_DIR = os.path.join(DATA_DIR, "test", "REAL")
print(f"CIFAKE REAL dir: {CIFAKE_REAL_DIR} (exists={os.path.exists(CIFAKE_REAL_DIR)})")


## 2. Download Real Control Images

Streams ~300 high-resolution real photographs from COCO 2017 val plus matched-source
reals from Defactify and CommunityForensics. Idempotent — skips folders that already
have enough images.

In [ ]:
# Cell 5 — Download high-res real controls
from scripts.download_real_controls import download_all_controls

N_IMAGES = 300

control_results = download_all_controls(
    output_dir=CONTROLS_DIR,
    n=N_IMAGES,
    seed=42,
)

# Prefer COCO as the primary high-res real set; fall back to others
hires_candidates = [
    os.path.join(CONTROLS_DIR, "coco_real", "REAL"),
    os.path.join(CONTROLS_DIR, "defactify_real", "REAL"),
    os.path.join(CONTROLS_DIR, "communityforensics_real", "REAL"),
]
HIRES_REAL_DIR = None
for cand in hires_candidates:
    if os.path.isdir(cand) and len(os.listdir(cand)) >= 50:
        HIRES_REAL_DIR = cand
        break

if HIRES_REAL_DIR is None:
    raise FileNotFoundError(
        "No usable high-res real control folder found. "
        "Re-run the download cell or place images manually under "
        f"{CONTROLS_DIR}/coco_real/REAL/"
    )
print(f"\nUsing high-res real dir: {HIRES_REAL_DIR}")
print(f"Image count: {len(os.listdir(HIRES_REAL_DIR))}")


## 3. Run the 2×2 Resolution Control Grid

In [ ]:
# Cell 6 — Discover generator FAKE folders and run the control
from src.utils.data_loader import discover_generator_families
from src.analysis.resolution_control import run_resolution_control

family_dirs = discover_generator_families(GEN_DIR, min_fake=50)
# Exclude _controls itself if discover picks it up (it shouldn't — no FAKE/)
generator_fake_dirs = {}
for fdir in family_dirs:
    name = Path(fdir).name
    if name.startswith("_"):
        continue
    fake = os.path.join(fdir, "FAKE")
    if os.path.isdir(fake):
        generator_fake_dirs[name] = fake

print(f"Generator families ({len(generator_fake_dirs)}):")
for name, path in sorted(generator_fake_dirs.items()):
    print(f"  {name:20s} | {len(os.listdir(path))} images")

results = run_resolution_control(
    model=model,
    device=device,
    cifake_real_dir=CIFAKE_REAL_DIR,
    hires_real_dir=HIRES_REAL_DIR,
    generator_fake_dirs=generator_fake_dirs,
    temperature=TEMPERATURE,
    n_images=N_IMAGES,
    batch_size=32,
    output_dir=OUTPUT_DIR,
)


## 4. Results Summary & Interpretation

In [ ]:
# Cell 7 — Pretty-print summary table
import pandas as pd
from IPython.display import display

conds = results["conditions"]
rows = [
    {
        "Condition": "A: CIFAKE REAL (native)",
        "True label": "REAL",
        "N": conds["A_cifake_real_native"]["n_images"],
        "FAKE-rate": conds["A_cifake_real_native"]["fake_rate"],
        "Mean P(FAKE)": conds["A_cifake_real_native"]["mean_p_fake"],
        "Median P(FAKE)": conds["A_cifake_real_native"]["median_p_fake"],
    },
    {
        "Condition": "B: Hi-res REAL (native)",
        "True label": "REAL",
        "N": conds["B_hires_real_native"]["n_images"],
        "FAKE-rate": conds["B_hires_real_native"]["fake_rate"],
        "Mean P(FAKE)": conds["B_hires_real_native"]["mean_p_fake"],
        "Median P(FAKE)": conds["B_hires_real_native"]["median_p_fake"],
    },
    {
        "Condition": "C: Hi-res REAL (→32×32)",
        "True label": "REAL",
        "N": conds["C_hires_real_matched"]["n_images"],
        "FAKE-rate": conds["C_hires_real_matched"]["fake_rate"],
        "Mean P(FAKE)": conds["C_hires_real_matched"]["mean_p_fake"],
        "Median P(FAKE)": conds["C_hires_real_matched"]["median_p_fake"],
    },
]

for fam, r in sorted(conds.get("D_generator_fakes_matched", {}).items()):
    native = conds.get("generator_fakes_native", {}).get(fam, {})
    rows.append({
        "Condition": f"D: {fam} FAKE (→32×32)",
        "True label": "FAKE",
        "N": r["n_images"],
        "FAKE-rate": r["fake_rate"],
        "Mean P(FAKE)": r["mean_p_fake"],
        "Median P(FAKE)": r["median_p_fake"],
    })
    if native:
        rows.append({
            "Condition": f"   {fam} FAKE (native)",
            "True label": "FAKE",
            "N": native["n_images"],
            "FAKE-rate": native["fake_rate"],
            "Mean P(FAKE)": native["mean_p_fake"],
            "Median P(FAKE)": native["median_p_fake"],
        })

df = pd.DataFrame(rows)
display(
    df.style.format({
        "FAKE-rate": "{:.1%}",
        "Mean P(FAKE)": "{:.4f}",
        "Median P(FAKE)": "{:.4f}",
    }).hide(axis="index")
)

interp = results["interpretation"]
print("\n=== INTERPRETATION ===")
print(f"  A (CIFAKE real) FAKE-rate:     {interp['A_cifake_real_fake_rate']:.1%}")
print(f"  B (hi-res real) FAKE-rate:     {interp['B_hires_real_fake_rate']:.1%}")
print(f"  C (hi-res→32×32) FAKE-rate:    {interp['C_hires_real_matched_fake_rate']:.1%}")
print(f"  Confound suspected:           {interp['resolution_confound_suspected']}")
print(f"\n  {interp['summary']}")


In [ ]:
# Cell 8 — Display saved plots
from IPython.display import Image as IPImage, display

plots_dir = os.path.join(OUTPUT_DIR, "plots", "resolution_control")
for name in [
    "fake_rate_by_condition.png",
    "p_fake_distributions.png",
    "before_after_resolution_matching.png",
]:
    path = os.path.join(plots_dir, name)
    print(f"\n--- {name} ---")
    if os.path.exists(path):
        display(IPImage(filename=path, width=700))
    else:
        print(f"  Missing: {path}")

print("\n" + "=" * 60)
print("RESOLUTION CONTROL COMPLETE")
print("=" * 60)
print(f"Results JSON: {os.path.join(OUTPUT_DIR, 'results', 'resolution_control.json')}")
print(f"Plots:        {plots_dir}")
print("\nCopy these files into the local repo under outputs/ before writing the final report.")


# 07 — Resolution-Matched Control Experiment

**Milestone 6** (August 11–28, 2026)

This notebook tests whether the cross-generator evaluation results are confounded by
resolution differences between CIFAKE (32×32 upscaled) and generator families (high-res
downscaled).

**Design:** A 2×2 grid over `{real, fake} × {native pipeline, force-to-32×32 pipeline}`:
- **A** CIFAKE REAL, standard eval pipeline (reference)
- **B** High-res real photos, standard eval pipeline (tests false-positive rate)
- **C** High-res real photos, force-to-32×32 pipeline (isolates resolution)
- **D** Generator fakes, force-to-32×32 pipeline (tests detection at matched resolution)

## 0. CIFAKE Setup

In [ ]:
# Cell 1 — Mount Google Drive & define paths
import os
from google.colab import drive

drive.mount("/content/drive")

DRIVE_BASE = "/content/drive/MyDrive/ai-image-detection"
DATA_DIR   = os.path.join(DRIVE_BASE, "data/raw/cifake")
CHECKPOINT_DIR = os.path.join(DRIVE_BASE, "checkpoints")

# Clone repo
if not os.path.exists("/content/ai-image-detection"):
    !git clone https://github.com/krishi-shah/ai-image-detection.git /content/ai-image-detection
os.chdir("/content/ai-image-detection")

print(f"Working directory: {os.getcwd()}")
print(f"Data directory:    {DATA_DIR}")
print(f"Checkpoint dir:    {CHECKPOINT_DIR}")

In [ ]:
# Cell 2 — Install dependencies
!pip install -q -r requirements.txt

In [ ]:
# Cell 3 — Extract CIFAKE from zip uploaded to Google Drive
import os, zipfile, glob

os.makedirs(DATA_DIR, exist_ok=True)

if not os.path.exists(os.path.join(DATA_DIR, "train")):
    zip_patterns = [
        os.path.join(DRIVE_BASE, "archive*.zip"),
        os.path.join(DRIVE_BASE, "cifake*.zip"),
    ]
    zip_path = None
    for pat in zip_patterns:
        matches = sorted(glob.glob(pat))
        if matches:
            zip_path = matches[-1]
            break

    if zip_path:
        print(f"Found zip: {zip_path}")
        print("Extracting (this takes ~1-2 minutes)...")
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(DATA_DIR)
        print("Extraction complete.")
    else:
        print("ERROR: No CIFAKE zip found on Drive.")
        print("Upload archive.zip to:", DRIVE_BASE)

print(f"Contents of DATA_DIR: {os.listdir(DATA_DIR)}")

## 1. Load Model

In [ ]:
import sys
import json
import torch

IN_COLAB = 'google.colab' in sys.modules
DRIVE_BASE_PATH = DRIVE_BASE if IN_COLAB else None

REAL_REFERENCE_DIR = os.path.join(DATA_DIR, 'test', 'REAL')
GEN_DATA_DIR = os.path.join(DRIVE_BASE, 'data/generalisation') if IN_COLAB else 'data/generalisation'
CONTROLS_DIR = os.path.join(DRIVE_BASE, 'data/generalisation/_controls') if IN_COLAB else 'data/generalisation/_controls'
OUTPUT_DIR = os.path.join(DRIVE_BASE, 'outputs') if IN_COLAB else 'outputs'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

from src.model.detector import build_detector, load_checkpoint

CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, 'best_detector.pth')
model = build_detector(pretrained=False)
load_checkpoint(model, CHECKPOINT_PATH)
model = model.to(device)
model.eval()

# Load baseline results for temperature
BASELINE_PATH = os.path.join(OUTPUT_DIR, 'results', 'baseline_results.json')
with open(BASELINE_PATH) as f:
    baseline_results = json.load(f)
TEMPERATURE = baseline_results.get('temperature', 1.0)

print(f"Model loaded. Baseline accuracy: {baseline_results['test_accuracy']}")
print(f"Temperature: {TEMPERATURE}")

## 2. Download Real Control Images

Download 300 high-resolution real photographs from each source:
- COCO 2017 validation (primary control)
- Defactify real images (matched source for SD3/Flux and Midjourney)
- CommunityForensics real images (matched source for StyleGAN)

In [ ]:
from scripts.download_real_controls import download_all_controls

control_results = download_all_controls(
    output_dir='data/generalisation/_controls',
    count=300,
    seed=42,
    base_path=DRIVE_BASE if IN_COLAB else None,
)

In [ ]:
# Verify downloads
from pathlib import Path
controls_root = Path(CONTROLS_DIR)
print("\nControl image counts:")
for source_dir in sorted(controls_root.iterdir()):
    if source_dir.is_dir():
        real_dir = source_dir / 'REAL'
        if real_dir.exists():
            count = sum(1 for f in real_dir.iterdir() if f.suffix.lower() in {'.png', '.jpg', '.jpeg'})
            print(f"  {source_dir.name:30s} | {count} images")

## 3. Run Resolution Control Experiment

Run the 2×2 grid using COCO real photographs as the primary high-resolution control.

In [ ]:
from src.analysis.resolution_control import run_resolution_control
from src.utils.data_loader import discover_generator_families

# Identify generator family FAKE directories
family_dirs = discover_generator_families(GEN_DATA_DIR)
generator_fake_dirs = {}
for fdir in family_dirs:
    family_name = Path(fdir).name
    fake_subdir = Path(fdir) / 'FAKE'
    if fake_subdir.exists():
        generator_fake_dirs[family_name] = str(fake_subdir)

print(f"Generator families: {list(generator_fake_dirs.keys())}")
print(f"CIFAKE REAL reference: {REAL_REFERENCE_DIR}")

# Use COCO as primary high-res real control
HIRES_REAL_DIR = str(controls_root / 'coco_real' / 'REAL')
print(f"High-res real control: {HIRES_REAL_DIR}")

In [ ]:
# Run the full 2x2 grid
results = run_resolution_control(
    model=model,
    device=device,
    cifake_real_dir=REAL_REFERENCE_DIR,
    hires_real_dir=HIRES_REAL_DIR,
    generator_fake_dirs=generator_fake_dirs,
    temperature=TEMPERATURE,
    output_dir=OUTPUT_DIR,
    max_images=300,
    batch_size=32,
)

## 4. Results Summary

In [ ]:
import pandas as pd

summary_rows = []
for key, res in results.items():
    if res.get('n', 0) == 0:
        continue
    summary_rows.append({
        'Condition': key,
        'True Label': res['true_label_name'],
        'N': res['n'],
        'FAKE-rate': f"{res['fake_rate']:.3f}",
        'Mean P(FAKE)': f"{res['mean_p_fake']:.4f}",
        'Median P(FAKE)': f"{res['median_p_fake']:.4f}",
    })

df_summary = pd.DataFrame(summary_rows)
display(df_summary.style.set_caption('Resolution Control Results'))

In [ ]:
# LaTeX table for final report
print("\n--- LaTeX table ---\n")
print(df_summary.to_latex(index=False, escape=True))

## 5. Visualisations

In [ ]:
from IPython.display import Image as IPImage, display as ipdisplay
import os

plots_dir = os.path.join(OUTPUT_DIR, 'plots', 'resolution_control')
for plot_name in [
    'fake_rate_by_condition.png',
    'p_fake_distributions.png',
    'family_resolution_comparison.png',
]:
    path = os.path.join(plots_dir, plot_name)
    if os.path.exists(path):
        print(f"\n--- {plot_name} ---")
        ipdisplay(IPImage(filename=path))

## 6. Matched-Source Controls (Optional)

Re-run conditions B and C using the matched-source real images from Defactify
and CommunityForensics (if available) for a tighter domain control.

In [ ]:
from src.analysis.resolution_control import evaluate_condition, get_native_eval_transform, get_matched_lowres_transform

matched_results = {}
native_tf = get_native_eval_transform()
lowres_tf = get_matched_lowres_transform()

for source_name in ['defactify_real', 'communityforensics_real']:
    real_dir = str(controls_root / source_name / 'REAL')
    if not os.path.exists(real_dir) or len(os.listdir(real_dir)) == 0:
        print(f"Skipping {source_name} (not available)")
        continue

    print(f"\n=== {source_name} ===")

    # Native pipeline
    r_native = evaluate_condition(
        model, real_dir, true_label=1, transform=native_tf,
        device=device, temperature=TEMPERATURE, max_images=300,
    )
    print(f"  Native pipeline:  FAKE-rate = {r_native['fake_rate']:.3f}, Mean P(FAKE) = {r_native['mean_p_fake']:.4f}")

    # Force-to-32x32 pipeline
    r_lowres = evaluate_condition(
        model, real_dir, true_label=1, transform=lowres_tf,
        device=device, temperature=TEMPERATURE, max_images=300,
    )
    print(f"  Force-32x32:      FAKE-rate = {r_lowres['fake_rate']:.3f}, Mean P(FAKE) = {r_lowres['mean_p_fake']:.4f}")

    matched_results[f"{source_name}_native"] = r_native
    matched_results[f"{source_name}_lowres"] = r_lowres

## 7. Discussion

**Interpretation guide:**

- If Condition B (high-res REAL, native pipeline) shows a **high** false-positive rate,
  the detector is biased by resolution — it misclassifies high-res real images as FAKE
  because it learned resolution-dependent features from CIFAKE's 32×32 images.

- If Condition B shows a **low** false-positive rate (comparable to Condition A),
  the detector genuinely transfers across resolutions and the cross-generator results
  are not confounded.

- Condition D reveals whether fake detection survives when the resolution advantage
  is removed by forcing fakes through the same 32×32 bottleneck as CIFAKE.

In [ ]:
print("\n" + "="*60)
print("RESOLUTION CONTROL COMPLETE")
print("="*60)
print(f"\nResults: {os.path.join(OUTPUT_DIR, 'results', 'resolution_control.json')}")
print(f"Plots:   {os.path.join(OUTPUT_DIR, 'plots', 'resolution_control', '')}")